# DTClassifier


In [8]:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score


In [9]:
# CONFIG
input_csv = "AllCars.csv"
test_size = 0.2
random_state = 42


In [10]:
# Load data
dataset = pd.read_csv(input_csv)

required = {"Volume","Doors","Style"}
missing = required - set(dataset.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

X = dataset[["Volume","Doors"]].copy()
X = X.apply(pd.to_numeric, errors="coerce")
med = X.median(numeric_only=True)
X = X.fillna(med)

y = dataset["Style"].astype(str).str.strip()


In [11]:
# Train/test split + train decision tree + predict
stratify = y if y.nunique() > 1 else None
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=random_state, stratify=stratify
)

# max_depth is set for readability of TreeCars.png
model = DecisionTreeClassifier(random_state=random_state, max_depth=4)
model.fit(X_train, y_train)

pred = model.predict(X_test)
acc = float(accuracy_score(y_test, pred))

acc


0.6875

In [14]:
# Save TreeCars.png (readable)
plt.figure(figsize=(24,12))
plot_tree(model, feature_names=["Volume","Doors"], class_names=sorted(y.unique()), filled=True, rounded=True, fontsize=12)
plt.tight_layout()
plt.savefig("TreeCars.png", dpi=400)
plt.close()


In [13]:
# Save TreeCars.csv with accuracy row
out = X_test.copy()
out["Style"] = y_test.to_numpy()
out["PredictedStyle"] = pred
out = out[["Volume","Doors","Style","PredictedStyle"]].reset_index(drop=True)

out = pd.concat([out, pd.DataFrame([{"Volume":"","Doors":"","Style":"Accuracy","PredictedStyle":acc}])], ignore_index=True)
out.to_csv("TreeCars.csv", index=False)

out.head()


,Volume,Doors,Style,PredictedStyle
0,14,4,Sedan,Sedan
1,101,4,Sedan,Sedan
2,122,5,SUV,SUV
3,128,4,SUV,SUV
4,168,5,SUV,SUV
